In [1]:
import os
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image

from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
PROJECT_ROOT = Path.cwd().parent
RAW_DATA = PROJECT_ROOT / "data" / "raw" / "Revitsone-5classes"
TRAIN_DIR = PROJECT_ROOT / "data" / "train"
VAL_DIR = PROJECT_ROOT / "data" / "validation"
TEST_DIR = PROJECT_ROOT / "data" / "test"

In [ ]:
print(RAW_DATA.exists())
print(list(RAW_DATA.iterdir()))


True
[PosixPath('/Users/admin/Desktop/Driver-Behavior-Detection-CNN/data/raw/Revitsone-5classes/texting_phone'), PosixPath('/Users/admin/Desktop/Driver-Behavior-Detection-CNN/data/raw/Revitsone-5classes/other_activities'), PosixPath('/Users/admin/Desktop/Driver-Behavior-Detection-CNN/data/raw/Revitsone-5classes/talking_phone'), PosixPath('/Users/admin/Desktop/Driver-Behavior-Detection-CNN/data/raw/Revitsone-5classes/safe_driving'), PosixPath('/Users/admin/Desktop/Driver-Behavior-Detection-CNN/data/raw/Revitsone-5classes/turning')]


In [4]:
bad_images = []

for class_name in os.listdir(RAW_DATA):

    class_path = RAW_DATA / class_name

    if not class_path.is_dir():
        continue

    for image_name in os.listdir(class_path):

        image_path = class_path / image_name

        try:
            img = Image.open(image_path)
            img.verify()

        except Exception:

            bad_images.append(image_path)

print("Corrupted Images:", len(bad_images))

Corrupted Images: 15


In [5]:
for image in bad_images:
    os.remove(image)

print("Removed!")


Removed!


In [6]:
data = []

for class_name in os.listdir(RAW_DATA):

    class_path = RAW_DATA / class_name

    if not class_path.is_dir():
        continue

    for image_name in os.listdir(class_path):

        image_path = class_path / image_name

        data.append(
            {
                "filepath": str(image_path),
                "label": class_name
            }
        )

df = pd.DataFrame(data)

df.head()

,filepath,label
0,/Users/admin/Desktop/Driver-Behavior-Detection...,texting_phone
1,/Users/admin/Desktop/Driver-Behavior-Detection...,texting_phone
2,/Users/admin/Desktop/Driver-Behavior-Detection...,texting_phone
3,/Users/admin/Desktop/Driver-Behavior-Detection...,texting_phone
4,/Users/admin/Desktop/Driver-Behavior-Detection...,texting_phone


In [7]:
print(df.shape)

(10751, 2)


In [8]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

In [9]:
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

In [10]:
print(len(train_df))
print(len(val_df))
print(len(test_df))

7525
1613
1613


In [11]:
train_datagen = ImageDataGenerator(
    rescale=1./255,

    rotation_range=20,

    zoom_range=0.2,

    width_shift_range=0.2,

    height_shift_range=0.2,

    horizontal_flip=True
)

In [12]:
test_datagen = ImageDataGenerator(
    rescale=1./255
)

In [13]:
IMAGE_SIZE = (224,224)

BATCH_SIZE = 32

In [14]:
train_generator = train_datagen.flow_from_dataframe(
    train_df,
    x_col="filepath",
    y_col="label",
    target_size=IMAGE_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True
)

Found 7525 validated image filenames belonging to 5 classes.


In [15]:
validation_generator = test_datagen.flow_from_dataframe(
    val_df,
    x_col="filepath",
    y_col="label",
    target_size=IMAGE_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 1613 validated image filenames belonging to 5 classes.


In [16]:
test_generator = test_datagen.flow_from_dataframe(
    test_df,
    x_col="filepath",
    y_col="label",
    target_size=IMAGE_SIZE,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False
)

Found 1613 validated image filenames belonging to 5 classes.


In [17]:
images, labels = next(train_generator)

print(images.shape)
print(labels.shape)

(32, 224, 224, 3)
(32, 5)
